In [ ]:
#0. MERFISH preprocessing steps
##Part A — Split large images into tiled blocks for downstream nucleus segmentation (CellProfiler)
##Part B — Stitch block masks, detect nucleus-containing cells, and match transcripts to cells.

In [ ]:
"""
Part A — Split large images into tiled blocks for downstream nucleus segmentation (CellProfiler)

Usage:
  - Edit CONFIG paths below
  - Run this script once (it writes block TIFFs and blocks_info.csv)
  - Then run CellProfiler with a pipeline that reads the DAPI blocks folder and writes a mask file
    for each block. See instructions below.
"""

import os
import tifffile
import numpy as np
import pandas as pd

# ---------------- CONFIG (EDIT) ----------------
DATA_DIR = os.path.abspath(os.environ.get("SPATIAL_VELO_RAW_DATA_DIR", os.path.join("..", "data", "raw")))
DAPI_TIF = os.path.join(DATA_DIR, "datasets_mouse_brain_map_BrainReceptorShowcase_Slice1_Replicate1_images_mosaic_DAPI_z0.tif")
POLYT_TIF = os.path.join(DATA_DIR, "datasets_mouse_brain_map_BrainReceptorShowcase_Slice1_Replicate1_images_mosaic_PolyT_z0.tif")
OUT_BLOCK_DIR = os.path.abspath(os.environ.get("SPATIAL_VELO_OUTPUT_DIR", os.path.join("..", "results", "step0_blocks")))   # will contain subfolders 'dapi' and 'polyt'
BLOCK_SIZE = 4096   # tile size in pixels (adjust to memory/CellProfiler preferences)
# ------------------------------------------------

def split_image(image_path, out_dir, block_size=4096, prefix="block"):
    """Split a TIFF into non-overlapping blocks and save them. Return list of block info dicts."""
    os.makedirs(out_dir, exist_ok=True)
    img = tifffile.imread(image_path)
    H, W = img.shape[0], img.shape[1]
    blocks = []
    for y in range(0, H, block_size):
        for x in range(0, W, block_size):
            block = img[y:y+block_size, x:x+block_size].copy()
            fname = f"{prefix}_{y}_{x}.tif"
            out_path = os.path.join(out_dir, fname)
            tifffile.imwrite(out_path, block)
            blocks.append({
                "file": fname,
                "y_start": y,
                "x_start": x,
                "y_end": y + block.shape[0],
                "x_end": x + block.shape[1]
            })
    return blocks

if __name__ == "__main__":
    dapi_dir = os.path.join(OUT_BLOCK_DIR, "dapi_blocks")
    polyt_dir = os.path.join(OUT_BLOCK_DIR, "polyt_blocks")
    print("Splitting DAPI image into blocks...")
    dapi_blocks = split_image(DAPI_TIF, dapi_dir, block_size=BLOCK_SIZE, prefix="dapi_block")
    print(f"Saved {len(dapi_blocks)} DAPI blocks to {dapi_dir}")
    print("Splitting PolyT image into blocks...")
    polyt_blocks = split_image(POLYT_TIF, polyt_dir, block_size=BLOCK_SIZE, prefix="polyt_block")
    print(f"Saved {len(polyt_blocks)} PolyT blocks to {polyt_dir}")
    # save block info tables for downstream stitching
    pd.DataFrame(dapi_blocks).to_csv(os.path.join(dapi_dir, "blocks_info.csv"), index=False)
    pd.DataFrame(polyt_blocks).to_csv(os.path.join(polyt_dir, "blocks_info.csv"), index=False)
    print("Saved blocks_info.csv in each blocks folder.")
    print("\nNEXT STEP: Run CellProfiler (or other segmentation tool) to detect nuclei.")
    print("Recommendation for CellProfiler:")
    print("  - Input images: DAPI block folder (dapi_blocks).")
    print("  - Pipeline: IdentifyPrimaryObjects on DAPI -> produce binary mask per block.")
    print("  - Save mask images with same basename and suffix '_mask.tif', e.g. 'dapi_block_0_0_mask.tif'.")
print("  - Save masks into the same 'dapi_blocks' folder (or a separate masks folder; update Part B accordingly).")